# Prediction API — Testes

Testa todos os endpoints de `/prediction`:
- `POST /prediction/predict` — Predição (sem persistência)
- `POST /prediction/predict_and_persist` — Predição com persistência (background)
- `GET /prediction/backtest/{veiculo_id}?target=` — Backtest

In [2]:
import httpx
import time

BASE_URL = "http://localhost:8010"
client = httpx.Client(base_url=BASE_URL, timeout=300)

## 1. Predição para veículos específicos (sem persistência)

In [3]:
TARGET = "km"
VEHICLE_IDS = [101, 102, 103]

resp = client.post("/prediction/predict", json={"target": TARGET, "vehicle_ids": VEHICLE_IDS})
print(f"Status: {resp.status_code}")
data = resp.json()
data.keys()

Status: 200


dict_keys(['target', 'predictions_daily', 'predictions_heads', 'type_probabilities', 'not_found'])

In [4]:
# Inspecionar predições daily
import pandas as pd

if data.get("predictions_daily"):
    df_daily = pd.DataFrame(data["predictions_daily"])
    display(df_daily.head(10))
else:
    print("Sem predições daily")

Sem predições daily


In [ ]:
# Inspecionar predições heads
if data.get("predictions_heads"):
    df_heads = pd.DataFrame(data["predictions_heads"])
    display(df_heads.head(10))
else:
    print("Sem predições heads")

In [ ]:
# Veículos não encontrados
if data.get("not_found"):
    pd.DataFrame(data["not_found"])
else:
    print("Todos os veículos encontrados")

## 2. Predição para todos os veículos (sem persistência)

In [ ]:
resp = client.post("/prediction/predict", json={"target": TARGET})
print(f"Status: {resp.status_code}")
data_all = resp.json()

print(f"Daily: {len(data_all.get('predictions_daily', []))} registos")
print(f"Heads: {len(data_all.get('predictions_heads', []))} registos")
print(f"Type probs: {len(data_all.get('type_probabilities', []))} veículos")
print(f"Not found: {len(data_all.get('not_found', []))} veículos")

## 3. Predição com persistência (background)

Submete a predição em background. Resposta imediata com 202.

In [ ]:
resp = client.post("/prediction/predict_and_persist", json={"target": TARGET})
print(f"Status: {resp.status_code}")
resp.json()

In [ ]:
# Testar conflito: submeter novamente enquanto a anterior corre
resp = client.post("/prediction/predict_and_persist", json={"target": TARGET})
print(f"Status: {resp.status_code}  (esperado: 409 se ainda em execução, 202 se já terminou)")
resp.json()

## 4. Backtest

Compara predições persistidas com valores reais de `daily_activity`.

**Requer** que `predict_and_persist` já tenha sido executado.

In [ ]:
VEICULO_ID = 101

resp = client.get(f"/prediction/backtest/{VEICULO_ID}", params={"target": TARGET})
print(f"Status: {resp.status_code}")
bt = resp.json()
bt.keys()

In [ ]:
# Daily: comparação data a data
if bt.get("daily"):
    df_bt_daily = pd.DataFrame(bt["daily"])
    df_bt_daily["error"] = df_bt_daily["actual"] - df_bt_daily["predicted"]
    display(df_bt_daily)
else:
    print("Sem dados daily")

In [ ]:
# Heads: comparação por horizonte
if bt.get("heads"):
    df_bt_heads = pd.DataFrame(bt["heads"])
    df_bt_heads["error"] = df_bt_heads["actual"] - df_bt_heads["predicted"]
    display(df_bt_heads)
else:
    print("Sem dados heads")

In [ ]:
# Testar backtest com veículo sem predições → 404
resp = client.get("/prediction/backtest/999999", params={"target": TARGET})
print(f"Status: {resp.status_code}")
resp.json()

## 5. Testar com target 'h'

In [ ]:
resp = client.post("/prediction/predict", json={"target": "h", "vehicle_ids": [101]})
print(f"Status: {resp.status_code}")
resp.json()